# 🧴 Skincare Stack — Skin Type Classifier

Notebook ini melatih model MobileNetV2 untuk mendeteksi tipe kulit:
**Oily (Berminyak)**, **Dry (Kering)**, **Normal**

## Langkah-langkah:
1. Pastikan Runtime → Change runtime type → **GPU (T4)**
2. Jalankan setiap cell dari atas ke bawah
3. Di Cell 3, upload file `kaggle.json` dari akun Kaggle kamu
4. Setelah selesai, download 2 file hasil: `skin_classifier.tflite` dan `skin_labels.txt`
5. Copy kedua file ke folder `assets/models/` di project Flutter


## Cell 1 — Install Library

In [ ]:
!pip install -q kaggle
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

print(f'✅ TensorFlow: {tf.__version__}')
print(f'✅ GPU: {"Available" if len(tf.config.list_physical_devices("GPU")) > 0 else "Not found — go to Runtime > Change runtime type > GPU"}')


## Cell 2 — Setup Kaggle API

Cara mendapatkan `kaggle.json`:
1. Buka [kaggle.com](https://www.kaggle.com) → klik foto profil → **Settings**
2. Scroll ke bagian **API** → klik **Create New Token**
3. File `kaggle.json` akan otomatis terdownload
4. Upload file tersebut ketika diminta di bawah


In [ ]:
from google.colab import files

print('📁 Upload file kaggle.json kamu:')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(list(uploaded.values())[0])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle API siap!')


## Cell 3 — Download Dataset dari Kaggle

In [ ]:
!kaggle datasets download -d shakyadissanayake/oily-dry-and-normal-skin-types-dataset -p ./skin_data --unzip -q

data_root = Path('./skin_data')
print('📂 Isi dataset:')
for path in sorted(data_root.rglob('*')):
    if path.is_dir():
        img_count = len(list(path.glob('*.jpg'))) + len(list(path.glob('*.png'))) + len(list(path.glob('*.jpeg')))
        if img_count > 0:
            print(f'   {str(path.relative_to(data_root))}: {img_count} gambar')

print('\n✅ Dataset berhasil didownload!')


## Cell 4 — Temukan Direktori Data

In [ ]:
def find_data_dir(root):
    classes = ['oily', 'dry', 'normal']
    # Cek root langsung
    if all((root / c).exists() for c in classes):
        return root
    # Cek subdirectory
    for subdir in sorted(root.rglob('oily')):
        if subdir.is_dir():
            parent = subdir.parent
            if all((parent / c).exists() for c in classes):
                return parent
    return None

data_dir = find_data_dir(data_root)

if data_dir:
    print(f'✅ Data directory: {data_dir}')
    for cls in ['oily', 'dry', 'normal']:
        count = len(list((data_dir / cls).glob('*.*')))
        print(f'   {cls}: {count} gambar')
else:
    print('❌ Folder tidak ditemukan. Cek struktur di bawah:')
    for p in sorted(data_root.rglob('*')):
        if p.is_dir():
            print(f'   {p}')
    raise ValueError('Ubah data_dir secara manual sesuai struktur yang tampil')


## Cell 5 — Siapkan Data Generator dengan Augmentasi

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.1,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
)

train_gen = datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=42,
    shuffle=True,
)

val_gen = datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=42,
    shuffle=False,
)

num_classes = len(train_gen.class_indices)
# Urutkan label sesuai index (penting untuk TFLite)
class_names = sorted(train_gen.class_indices, key=train_gen.class_indices.get)

print(f'\n📊 Dataset Summary:')
print(f'   Kelas: {class_names}')
print(f'   Training: {train_gen.samples} gambar')
print(f'   Validation: {val_gen.samples} gambar')


## Cell 6 — Bangun Model MobileNetV2

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # Freeze semua layer base

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print(f'✅ Model dibangun: MobileNetV2 + Custom Head')
print(f'   Parameters: {model.count_params():,}')
print(f'   Output classes: {num_classes} — {class_names}')


## Cell 7 — Training Phase 1: Classification Head

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
]

print('🚀 Phase 1: Training classification head (base model frozen)...')
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=callbacks,
    verbose=1,
)

best_acc = max(history1.history['val_accuracy'])
print(f'\n✅ Phase 1 selesai! Best val accuracy: {best_acc:.2%}')


## Cell 8 — Training Phase 2: Fine-tuning

In [ ]:
print('🔥 Phase 2: Fine-tuning top layers...')

base_model.trainable = True
# Freeze 100 layer pertama, unfreeze sisanya
for layer in base_model.layers[:100]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'   Layers yang di-unfreeze: {trainable_count}/{len(base_model.layers)}')

model.compile(
    optimizer=Adam(learning_rate=1e-5),  # LR lebih kecil untuk fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks,
    verbose=1,
)

val_loss, val_acc = model.evaluate(val_gen, verbose=0)
print(f'\n✅ Fine-tuning selesai!')
print(f'   Final Validation Accuracy: {val_acc:.2%}')
print(f'   Final Validation Loss:     {val_loss:.4f}')


## Cell 9 — Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gabungkan history
all_acc = history1.history['accuracy'] + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss = history1.history['loss'] + history2.history['loss']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']
epochs_range = range(1, len(all_acc) + 1)
phase2_start = len(history1.history['accuracy'])

axes[0].plot(epochs_range, all_acc, label='Train Accuracy', color='#3F51B5')
axes[0].plot(epochs_range, all_val_acc, label='Val Accuracy', color='#E91E63')
axes[0].axvline(x=phase2_start, color='gray', linestyle='--', label='Fine-tuning start')
axes[0].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, all_loss, label='Train Loss', color='#3F51B5')
axes[1].plot(epochs_range, all_val_loss, label='Val Loss', color='#E91E63')
axes[1].axvline(x=phase2_start, color='gray', linestyle='--', label='Fine-tuning start')
axes[1].set_title('Loss', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Skincare Stack — Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plot disimpan: training_history.png')


## Cell 10 — Konversi ke TFLite & Simpan Label

In [ ]:
# Simpan label sesuai urutan index model
with open('skin_labels.txt', 'w') as f:
    for label in class_names:
        f.write(label + '\n')
print(f'✅ Labels disimpan: {class_names}')

# Konversi ke TFLite dengan float16 quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open('skin_classifier.tflite', 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / (1024 * 1024)
print(f'✅ TFLite model disimpan!')
print(f'   Ukuran model: {size_mb:.2f} MB')


## Cell 11 — Test Model TFLite
Verifikasi model berjalan dengan benar sebelum download.


In [ ]:
import numpy as np

# Load model TFLite
interpreter = tf.lite.Interpreter(model_path='skin_classifier.tflite')
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('📋 Model Info:')
print(f'   Input shape:  {input_details[0]["shape"]} (expected: [1, 224, 224, 3])')
print(f'   Output shape: {output_details[0]["shape"]} (expected: [1, {num_classes}])')
print(f'   Input dtype:  {input_details[0]["dtype"]}')

# Test dengan dummy image
dummy_input = np.random.rand(1, 224, 224, 3).astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], dummy_input)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])

print(f'\n🧪 Test inference:')
for i, (name, prob) in enumerate(zip(class_names, output[0])):
    print(f'   {name}: {prob:.2%}')

print('\n✅ Model TFLite berfungsi dengan baik!')


## Cell 12 — Download File untuk Flutter

Download kedua file ini, lalu copy ke folder `assets/models/` di project Flutter kamu.


In [ ]:
from google.colab import files

print('📥 Mendownload file hasil training...')
print('\n⚠️  Setelah download, copy kedua file ini ke:')
print('   skincare_stack/assets/models/skin_classifier.tflite')
print('   skincare_stack/assets/models/skin_labels.txt')
print()

files.download('skin_classifier.tflite')
files.download('skin_labels.txt')

print('\n✅ Selesai! Jalankan flutter pub get lalu flutter run')
